In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,6)
 
 
# Ajustar path según estructura del proyecto
df = pd.read_csv("../Exploracion/BD_Adult_income/Adult_income_dataset.csv",header=0, na_values='?')
 
# Vista inicial de los datos
df.head()

In [ ]:
# =====================================
# Revisión de valores faltantes
# =====================================
print("Valores faltantes por columna:\n", df.isna().sum())
 
# Estrategia de tratamiento: reemplazo con 'Unknown' en categóricas
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')
print("\nValores faltantes después del tratamiento:\n", df.isna().sum())

In [ ]:
df.dtypes

In [ ]:
# =====================================
# Estadísticas descriptivas univariadas
# =====================================
# Variables numéricas
num_cols = df.select_dtypes(include=['int64','float64']).columns
df[num_cols].describe()
 
# Variables categóricas
df[categorical_cols].describe()

In [ ]:
# =====================================
# Detección de outliers (valores atípicos)
# =====================================
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers detectados")
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot de {col}')
    plt.show()

In [ ]:
# =====================================
# Análisis de distribuciones
# =====================================
for col in num_cols:
    plt.figure()
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribución de {col}')
    plt.show()
# Transformación logarítmica para capital-gain y capital-loss
df['capital-gain-log'] = np.log1p(df['capital-gain'])
df['capital-loss-log'] = np.log1p(df['capital-loss'])

In [ ]:
# =====================================
# Análisis univariado categórico
# =====================================
for col in categorical_cols:
    plt.figure()
    sns.countplot(y=df[col], order=df[col].value_counts().index)
    plt.title(f'Frecuencia de {col}')
    plt.show()

In [ ]:
# =====================================
# Análisis multivariado
# =====================================
# Mapa de calor de correlaciones
corr = df[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Mapa de calor de correlaciones")
plt.show()

# Scatter plots
sns.scatterplot(x='age', y='hours-per-week', hue='income', data=df)
plt.title("Horas por semana vs Edad")
plt.show()

sns.scatterplot(x='capital-gain', y='hours-per-week', hue='income', data=df)
plt.title("Capital gain vs Horas por semana")
plt.show()

# Pairplot variables seleccionadas
sns.pairplot(df[['age','educational-num','hours-per-week','capital-gain','capital-loss','income']], hue='income')
plt.show()

In [ ]:
# =====================================
# Tablas cruzadas y relaciones categórico-categórico
# =====================================
income_edu = pd.crosstab(df['education'], df['income'], normalize='index') * 100
print("Porcentaje de income por educación:\n", income_edu)

In [ ]:
# =====================================
# Codificación de variables categóricas
# =====================================
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Copia del dataset para trabajar
df_pre = df.copy()

# Identificar columnas categóricas
categorical_cols = df_pre.select_dtypes(include='object').columns
print("Columnas categóricas:", categorical_cols)

# One-Hot Encoding (recomendado)
df_encoded = pd.get_dummies(df_pre, columns=categorical_cols, drop_first=True)

print("Dimensión después de encoding:", df_encoded.shape)
df_encoded.head()